In [1]:
!pip install autoawq -q

In [2]:
!pip install git+https://github.com/huggingface/transformers.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip install huggingface_hub

from huggingface_hub import notebook_login

notebook_login()

zero_point : for false it create a symmetric layout with zero in the center while true it set 0 point according to weights that crucial for the ReLU because in ReLU or GeLU weights arent symmentix aroud zero

q_group_size : for 128 it take 128 weights and then apply the scale factor on that batch of 128 it make model more accurate

it comes with performance-accuracy tradeoff the small batch you take you got higher precision but file become large and exact reverce for the ;arge group size



In [6]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import torch

model_path = 'PY007/TinyLlama-1.1B-Chat-v0.3'

quant_name = model_path.split('/')[-1] + 'AWQ'
quant_path = 'AWQ/' + quant_name
quant_config = {'zero_point': True, 'q_group_size':128, 'w_bit':4}

# load model
model = AutoAWQForCausalLM.from_pretrained(model_path, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# quantize
model.quantize(tokenizer, quant_config)

# save model
model.save_quantized(quant_name, safetensors=True)
tokenizer.save_pretrained(quant_name)

print(f"Files saved to folder: {quant_name}")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.
AWQ: 100%|██████████| 22/22 [18:11<00:00, 49.59s/it]


Writing model shards: 0it [00:00, ?it/s]

Files saved to folder: TinyLlama-1.1B-Chat-v0.3AWQ


# save model to HF

In [7]:
from huggingface_hub import HfApi

api = HfApi()

# 1. Configuration
local_folder = "./" + quant_name  # The folder containing your safetensors and json files
repo_id = "mahendra4/TinyLlama-1.1B-Chat-v0.3AWQ"

# 2. Upload the entire folder
api.upload_folder(
    folder_path=local_folder,
    repo_id=repo_id,
    repo_type="model",
    # This ignores Colab's default sample folder and any hidden system files
    ignore_patterns=["sample_data/*", ".config/*", ".ipynb_checkpoints/*"]
)

print(f"✅ Success! View your model at: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...v0.3AWQ/model.safetensors:   2%|2         | 22.2MB / 1.03GB            

✅ Success! View your model at: https://huggingface.co/mahendra4/TinyLlama-1.1B-Chat-v0.3AWQ


In [11]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# Path to your model on Hugging Face
model_id = "mahendra4/TinyLlama-1.1B-Chat-v0.3AWQ"

# Load the Model

model = AutoAWQForCausalLM.from_quantized(model_id, fuse_layers=True, device_map="auto")

# Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# prompt
prompt = "Explain quantum computing in one sentence."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Replacing layers...: 100%|██████████| 22/22 [00:05<00:00,  4.01it/s]
/usr/local/lib/python3.12/dist-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed.No module named 'awq_ext'
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Explain quantum computing in one sentence.
What is quantum computing?
Quantum computing is a type of computing that uses quantum bits, or qubits, instead of classical bits. Unlike classical bits, which can only take on one value at a time, qubits can exist in multiple states at once. This allows quantum computers to perform calculations much faster than classical computers, especially in the realm of machine learning and artificial intelligence.
What are the key advantages of quantum computing?
Some of the key advantages of quantum computing include


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
